In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

DataFrame[]

In [0]:
transactions = spark.table("silver.transactions")
cards = spark.table("silver.cards")
users = spark.table("silver.users")

display(transactions.limit(5))

mcc,transaction_id,transaction_date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,fraud_label,merchant_category
5411,7475470,2010-01-01T03:26:00Z,609,2892,11.44,Swipe Transaction,46284,Garland,TX,75040,null,No,"Grocery Stores, Supermarkets"
5541,7475708,2010-01-01T06:41:00Z,840,37,87.0,Swipe Transaction,61195,San Francisco,CA,94131,null,null,Service Stations
4814,7475737,2010-01-01T06:49:00Z,119,5980,178.53,Online Transaction,73186,ONLINE,null,null,null,null,Telecommunication Services
5812,7475781,2010-01-01T07:00:00Z,980,5052,1.72,Swipe Transaction,63765,Gwynn Oak,MD,21207,null,null,Eating Places and Restaurants
5499,7476046,2010-01-01T08:03:00Z,1591,2043,-80.0,Swipe Transaction,59935,Omaha,NE,68106,null,No,Miscellaneous Food Stores


In [0]:
# Create Time Features

from pyspark.sql.functions import *

transactions_time = transactions \
.withColumn("date", to_date("transaction_date")) \
.withColumn("day_of_week", date_format("transaction_date","EEEE")) \
.withColumn("month", month("transaction_date")) \
.withColumn("week", weekofyear("transaction_date")) \
.withColumn("hour", hour("transaction_date")) \
.withColumn(
    "time_of_day",
    when(col("hour") < 12, "Morning")
    .when(col("hour") < 18, "Afternoon")
    .otherwise("Night")
) \
.withColumn(
    "is_fraud",
    when(col("fraud_label") == "Yes", True).otherwise(False)
)

display(transactions_time.limit(5))

mcc,transaction_id,transaction_date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,fraud_label,merchant_category,date,day_of_week,month,week,hour,time_of_day,is_fraud
7538,7475344,2010-01-01T00:32:00Z,646,2093,73.79,Swipe Transaction,1636,Erie,PA,16511,null,No,Automotive Service Shops,2010-01-01,Friday,1,53,0,Morning,false
5411,7475580,2010-01-01T06:02:00Z,1529,4985,3.98,Swipe Transaction,23492,Pompano Beach,FL,33076,null,No,"Grocery Stores, Supermarkets",2010-01-01,Friday,1,53,6,Morning,false
4722,7475623,2010-01-01T06:16:00Z,135,2808,-148.0,Online Transaction,90999,ONLINE,null,null,null,No,Travel Agencies,2010-01-01,Friday,1,53,6,Morning,false
5942,7475933,2010-01-01T07:36:00Z,1764,2118,3.38,Swipe Transaction,20519,Cadillac,MI,49601,null,null,Book Stores,2010-01-01,Friday,1,53,7,Morning,false
4829,7476461,2010-01-01T09:42:00Z,860,5412,40.0,Swipe Transaction,27092,Silver Spring,MD,20901,null,null,Money Transfer,2010-01-01,Friday,1,53,9,Morning,false


In [0]:
# Fraud Time Metrics

fraud_time_metrics = transactions_time \
.groupBy(
    "date",
    "day_of_week",
    "month",
    "week",
    "time_of_day"
) \
.agg(
    count("*").alias("total_transactions"),
    sum(col("is_fraud").cast("int")).alias("fraud_transactions"),
    sum(when(col("is_fraud")==True, col("amount"))).alias("fraud_loss"),
    avg("amount").alias("avg_transaction_amount")
) \
.withColumn(
    "fraud_rate",
    col("fraud_transactions") / col("total_transactions")
)

fraud_time_metrics.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.fraud_time_metrics")

display(fraud_time_metrics)

date,day_of_week,month,week,time_of_day,total_transactions,fraud_transactions,fraud_loss,avg_transaction_amount,fraud_rate
2010-07-06,Tuesday,7,27,Night,610,0,null,55.01672125186344,0.0
2010-09-04,Saturday,9,35,Afternoon,1186,0,null,44.96674547449986,0.0
2010-11-29,Monday,11,48,Night,573,0,null,50.90907515946899,0.0
2011-01-12,Wednesday,1,2,Afternoon,1332,0,null,43.15314559971315,0.0
2011-01-21,Friday,1,3,Morning,1610,0,null,39.35129808346215,0.0
2011-02-16,Wednesday,2,7,Night,637,0,null,46.83169551825318,0.0
2011-03-02,Wednesday,3,9,Afternoon,1258,0,null,44.88025438428423,0.0
2011-03-30,Wednesday,3,13,Morning,1610,0,null,37.1571055847249,0.0
2011-04-25,Monday,4,17,Afternoon,1327,0,null,48.163843171560146,0.0
2011-07-10,Sunday,7,27,Afternoon,1364,0,null,43.79206012349434,0.0


In [0]:
# Fraud User Metrics

fraud_user_metrics = transactions_time \
.groupBy("client_id","week") \
.agg(
    count("*").alias("total_transactions"),
    sum(col("is_fraud").cast("int")).alias("fraud_transactions"),
    avg("amount").alias("avg_transaction_amount"),
    max("amount").alias("max_transaction_amount")
)

fraud_user_metrics.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.fraud_user_metrics")

display(fraud_user_metrics)

client_id,week,total_transactions,fraud_transactions,avg_transaction_amount,max_transaction_amount
348,2,107,0,46.27682207900787,287.02
92,2,246,0,27.91256092331274,181.17
1171,5,146,7,84.45554795065155,1073.75
973,10,312,0,18.159230745373627,140.0
1138,10,354,0,35.73728807548345,860.98
39,11,272,0,56.468602701501155,546.99
1193,13,240,0,29.953041645884515,183.28
576,14,181,0,59.18469571342784,1091.08
1416,15,204,0,59.27642184379054,693.08
250,20,227,0,40.87726863943008,437.12


In [0]:
# Merchant Fraud Metrics

fraud_merchant_metrics = transactions_time \
.groupBy(
    "merchant_id",
    "merchant_category"
) \
.agg(
    count("*").alias("total_transactions"),
    sum(col("is_fraud").cast("int")).alias("fraud_transactions"),
    avg("amount").alias("avg_transaction_amount")
) \
.withColumn(
    "fraud_rate",
    col("fraud_transactions") / col("total_transactions")
)

fraud_merchant_metrics.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.fraud_merchant_metrics")

display(fraud_merchant_metrics)

merchant_id,merchant_category,total_transactions,fraud_transactions,avg_transaction_amount,fraud_rate
14528,Miscellaneous Food Stores,333505,4,1.321044091397778,1.1993823181061752E-5
92741,Drinking Places (Alcoholic Beverages),3454,0,19.64136365720421,0.0
27310,Cleaning and Maintenance Services,14020,2,41.23687375581873,1.426533523537803E-4
1715,"Grocery Stores, Supermarkets",82,0,20.3491464271778,0.0
18093,Wholesale Clubs,333,0,35.9749849522794,0.0
79262,Family Clothing Stores,41,0,74.32926841479976,0.0
69735,Automotive Service Shops,1320,0,3.5290606039491568,0.0
18696,"Doctors, Physicians",147,0,89.74238070014383,0.0
29220,Eating Places and Restaurants,826,0,12.990690077188228,0.0
25243,Wholesale Clubs,640,1,47.76021876693703,0.0015625


In [0]:
# Fraud Amount Segments

fraud_amount_metrics = transactions_time \
.withColumn(
    "amount_segment",
    when(col("amount") < 50, "Low")
    .when(col("amount") < 200, "Medium")
    .otherwise("High")
) \
.groupBy("amount_segment","is_fraud") \
.agg(
    count("*").alias("transactions"),
    avg("amount").alias("avg_amount")
)

fraud_amount_metrics.write.format("delta") \
.mode("overwrite") \
.saveAsTable("gold.fraud_amount_metrics")

display(fraud_amount_metrics)

amount_segment,is_fraud,transactions,avg_amount
High,true,2195,411.98236463542406
High,false,321982,362.0367608125968
Medium,true,5706,108.055683566527
Low,false,8841396,9.784810011925158
Medium,false,4129205,88.94815662408713
Low,true,5431,-9.43071994283673
